This notebook is based on the following work:

- https://muafzal.medium.com/unlock-microsoft-fabric-chargeback-data-from-built-in-reports-to-a-custom-lakehouse-and-finops-cdd8b449f3ac
- https://github.com/mafzal786/fabric-chargeback/blob/main/extract%20chargeback%20data.ipynb

Notebook adds the following capabilities:
- Store extracted CSV files under "Files/tableName" to keep historical record (ex.: chargeback_20261028.csv)
- Instead of overwriting destination delta table, only inserts new records based on existing data and keys that unique identify records in each table


In [ ]:
# Welcome to your new notebook
%pip install -U semantic-link semantic-link-sempy requests 

In [ ]:
import sempy.fabric as fabric
import requests
from notebookutils import credentials
import pandas as pd
import time
from datetime import datetime

In [ ]:
df_capacity = fabric.list_capacities()
print(df_capacity)

In [ ]:
df_ws = fabric.list_workspaces()
df_ws

In [ ]:
workspace_id= fabric.get_workspace_id()
print(workspace_id)

In [ ]:
items = fabric.list_items(workspace=workspace_id)
display(items)

In [ ]:
# get token
token = credentials.getToken(
    "https://analysis.windows.net/powerbi/api"
)

print(token[:100])

# add token to the header
headers = {
    "Authorization": f"Bearer {token}",
    "Content-Type": "application/json"
}

**Entity: Charge Back**

In [ ]:
item_name = "rpt_Chargeback"
dt_table_name = "chargeback"

# Filter the items DataFrame for the desired report
matching_items = items.loc[items["Display Name"] == item_name, "Id"]

# Validate that we found at least one matching item
if matching_items.empty:
    raise ValueError(f"No report found with Display Name == '{item_name}'. Check the name in the 'items' DataFrame.")

# Safely get the first matching report Id
report_id = matching_items.iloc[0]

# build report url
url = (
    f"https://api.powerbi.com/v1.0/myorg"
    f"/reports/{report_id}"
    f"/ExportTo"
)

print(url)

# create CSV
payload = {
    "format": "CSV"
}

headers = {
    "Authorization": f"Bearer {token}"
}

response = requests.post(
    url,
    headers=headers,
    json=payload
)
result = response.json()
print(result)

# Validate export response
if "id" not in result:
    raise RuntimeError(f"Export failed or unexpected response format: {result}")

# Get the export id
export_id = result["id"]

print(export_id)

# wait for export to be completed.
status_url = f"https://api.powerbi.com/v1.0/myorg/reports/{report_id}/exports/{export_id}"
while True:

    r = requests.get(
        status_url,
        headers=headers
    )

    result = r.json()

    print(result.get("status", "No status in response"))

    if result.get("status") == "Succeeded":
        break
    elif result.get("status") == "Failed":
        raise RuntimeError(f"Export failed: {result}")

    time.sleep(10)


# download the CSV

# Resource location from your export result
download_url = f"https://api.powerbi.com/v1.0/myorg/reports/{report_id}/exports/{export_id}/file"

print(download_url)

r = requests.get(
    download_url,
    headers=headers
)
print("download status:")
print(r.status_code)

if r.status_code != 200:
    raise RuntimeError(f"Download failed with status code {r.status_code}: {r.text}")

# Build dynamic filename: Chargeback_yyyyMMdd.csv
suffix = datetime.now().strftime("%Y%m%d")
file_name = f"Chargeback_{suffix}.csv"

# Save to Lakehouse Files area
output_file = f"/lakehouse/default/Files/Chargeback/{file_name}"
with open(output_file, "wb") as f:
    f.write(r.content)
print("File saved:", output_file)

# create a dataframe

df = pd.read_csv(
    output_file
)

display(df.head())

# create/append to delta table using composite key
spark_df = spark.createDataFrame(df)

from pyspark.sql.functions import col

key_cols = [
    "Chargeback_Capacity_Id",
    "Chargeback_Workspace_Id",
    "Chargeback_Date",
    "Chargeback_Item_Id",
    "Chargeback_Operation_name",
    "Chargeback_User",
]

if spark.catalog.tableExists(dt_table_name):
    print("Chargeback table exists")

    # Select the composite key from existing table
    existing_keys_df = spark.table(dt_table_name).select(*key_cols).distinct()

    # Join on all key columns to find only new rows, treating NULL = NULL via eqNullSafe
    join_condition = [spark_df[c].eqNullSafe(existing_keys_df[c]) for c in key_cols]
    new_rows_df = spark_df.join(existing_keys_df, join_condition, how="left_anti")

    inserted_count = new_rows_df.count()
    if inserted_count == 0:
        print(f"No new rows to insert into '{dt_table_name}'.")
    else:
        new_rows_df.write \
            .mode("append") \
            .format("delta") \
            .saveAsTable(dt_table_name)
        print(f"Inserted {inserted_count} new rows into '{dt_table_name}'.")
else:
    # First load: create the table with all rows
    print("Overwriting chargeback table")
    spark_df.write \
        .mode("overwrite") \
        .format("delta") \
        .saveAsTable(dt_table_name)
    print(f"table {dt_table_name} created")

print("chargeback load completed (insert-only for new composite keys)...")


**Entity: Workspaces**

In [ ]:
item_name = "rpt_Workspaces"

dt_table_name = "workspaces"

report_id = items.loc[
    items["Display Name"] == item_name,
    "Id"
].iloc[0]

# build report url
url = (
    f"https://api.powerbi.com/v1.0/myorg"
    f"/reports/{report_id}"
    f"/ExportTo"
)

print(url)

#create CSV

payload = {
    "format": "CSV"
}

headers = {
    "Authorization": f"Bearer {token}"
}

response = requests.post(
    url,
    headers=headers,
    json=payload
)
result = response.json()
print(result)

# Get the export id

export_id = result["id"]

print(export_id)

# wait for export to be completed.
status_url = f"https://api.powerbi.com/v1.0/myorg/reports/{report_id}/exports/{export_id}"

while True:

    r = requests.get(
        status_url,
        headers=headers
    )

    result = r.json()
    

    print(result["status"])

    if result["status"] == "Succeeded":
        break

    time.sleep(10)


# download the CSV

# Resource location from your export result
download_url = f"https://api.powerbi.com/v1.0/myorg/reports/{report_id}/exports/{export_id}/file"

print(download_url)

r = requests.get(
    download_url,
    headers=headers
)
print("download status:")
print(r.status_code)

# Save to Lakehouse Files area
# Build dynamic filename: Workspaces_yyyyMMdd.csv
suffix = datetime.now().strftime("%Y%m%d")
file_name = f"Workspaces_{suffix}.csv"

# Save to Lakehouse Files area
output_file = f"/lakehouse/default/Files/Workspaces/{file_name}"
with open(output_file, "wb") as f:
    f.write(r.content)
print("File saved:", output_file)

# create a dataframe

df = pd.read_csv(
    output_file
)

display(df.head())

# create/append to delta table with only new Workspaces_Workspace_Id rows
spark_df = spark.createDataFrame(df)

key_col = "Workspaces_Workspace_Id"
from pyspark.sql.functions import col

if spark.catalog.tableExists(dt_table_name):
    # Load existing table and find only new keys
    print("Workspaces table exists")
    existing_keys_df = spark.table(dt_table_name).select(key_col).distinct()
    new_rows_df = spark_df.join(existing_keys_df, on=key_col, how="left_anti")

    inserted_count = new_rows_df.count()
    if inserted_count == 0:
        print(f"No new rows to insert into '{dt_table_name}'.")
    else:
        new_rows_df.write \
            .mode("append") \
            .format("delta") \
            .saveAsTable(dt_table_name)
        print(f"Inserted {inserted_count} new rows into '{dt_table_name}'.")
else:
    # First load: create the table with all rows
    print("Overwriting workspaces table")
    spark_df.write \
        .mode("overwrite") \
        .format("delta") \
        .saveAsTable(dt_table_name)
    print(f"table {dt_table_name} created")

print("workspaces load completed (insert-only for new keys)...")    

**Entity: Items**

In [ ]:
item_name = "rpt_Items"

dt_table_name = "items"

report_id = items.loc[
    items["Display Name"] == item_name,
    "Id"
].iloc[0]

# build report url
url = (
    f"https://api.powerbi.com/v1.0/myorg"
    f"/reports/{report_id}"
    f"/ExportTo"
)

print(url)

#create CSV

payload = {
    "format": "CSV"
}

headers = {
    "Authorization": f"Bearer {token}"
}

response = requests.post(
    url,
    headers=headers,
    json=payload
)
result = response.json()
print(result)

# Get the export id

export_id = result["id"]

print(export_id)

# wait for export to be completed.
status_url = f"https://api.powerbi.com/v1.0/myorg/reports/{report_id}/exports/{export_id}"
while True:

    r = requests.get(
        status_url,
        headers=headers
    )

    result = r.json()

    print(result["status"])

    if result["status"] == "Succeeded":
        break

    time.sleep(10)


# download the CSV

# Resource location from your export result
download_url = f"https://api.powerbi.com/v1.0/myorg/reports/{report_id}/exports/{export_id}/file"

print(download_url)

r = requests.get(
    download_url,
    headers=headers
)
print("download status:")
print(r.status_code)

# Save to Lakehouse Files area
# Build dynamic filename: Items_yyyyMMdd.csv
suffix = datetime.now().strftime("%Y%m%d")
file_name = f"Items_{suffix}.csv"

# Save to Lakehouse Files area
output_file = f"/lakehouse/default/Files/Items/{file_name}"
with open(output_file, "wb") as f:
    f.write(r.content)
print("File saved:", output_file)

# create a dataframe

df = pd.read_csv(
    output_file
)

display(df.head())

# create/append to delta table with only new Items_Item_Id rows
spark_df = spark.createDataFrame(df)

key_col = "Items_Item_Id"
from pyspark.sql.functions import col

if spark.catalog.tableExists(dt_table_name):
    # Load existing table and find only new keys
    print("Items table exists")
    existing_keys_df = spark.table(dt_table_name).select(key_col).distinct()
    new_rows_df = spark_df.join(existing_keys_df, on=key_col, how="left_anti")

    inserted_count = new_rows_df.count()
    if inserted_count == 0:
        print(f"No new rows to insert into '{dt_table_name}'.")
    else:
        new_rows_df.write \
            .mode("append") \
            .format("delta") \
            .saveAsTable(dt_table_name)
        print(f"Inserted {inserted_count} new rows into '{dt_table_name}'.")
else:
    # First load: create the table with all rows
    print("Overwriting items table")
    spark_df.write \
        .mode("overwrite") \
        .format("delta") \
        .saveAsTable(dt_table_name)
    print(f"table {dt_table_name} created")

print("items load completed (insert-only for new keys)...")    

**Entity: Dates**

In [ ]:
item_name = "rpt_Dates"

dt_table_name = "dates"

report_id = items.loc[
    items["Display Name"] == item_name,
    "Id"
].iloc[0]

# build report url
url = (
    f"https://api.powerbi.com/v1.0/myorg"
    f"/reports/{report_id}"
    f"/ExportTo"
)

print(url)

#create CSV

payload = {
    "format": "CSV"
}

headers = {
    "Authorization": f"Bearer {token}"
}

response = requests.post(
    url,
    headers=headers,
    json=payload
)
result = response.json()
print(result)

# Get the export id

export_id = result["id"]

print(export_id)

# wait for export to be completed.
status_url = f"https://api.powerbi.com/v1.0/myorg/reports/{report_id}/exports/{export_id}"
while True:

    r = requests.get(
        status_url,
        headers=headers
    )

    result = r.json()

    print(result["status"])

    if result["status"] == "Succeeded":
        break

    time.sleep(10)


# download the CSV

# Resource location from your export result
download_url = f"https://api.powerbi.com/v1.0/myorg/reports/{report_id}/exports/{export_id}/file"

print(download_url)

r = requests.get(
    download_url,
    headers=headers
)
print("download status:")
print(r.status_code)

# Save to Lakehouse Files area

# Build dynamic filename: capacities_yyyyMMdd.csv
suffix = datetime.now().strftime("%Y%m%d")
file_name = f"Dates_{suffix}.csv"

# Save to Lakehouse Files area
output_file = f"/lakehouse/default/Files/Dates/{file_name}"
with open(output_file, "wb") as f:
    f.write(r.content)
print("File saved:", output_file)

# create a dataframe

df = pd.read_csv(
    output_file
)

display(df.head())

# create/append to delta table with only new Dates_Date rows
spark_df = spark.createDataFrame(df)

key_col = "Dates_Date"
from pyspark.sql.functions import col

if spark.catalog.tableExists(dt_table_name):
    # Load existing table and find only new keys
    print("Dates table exists")
    existing_keys_df = spark.table(dt_table_name).select(key_col).distinct()
    new_rows_df = spark_df.join(existing_keys_df, on=key_col, how="left_anti")

    inserted_count = new_rows_df.count()
    if inserted_count == 0:
        print(f"No new rows to insert into '{dt_table_name}'.")
    else:
        new_rows_df.write \
            .mode("append") \
            .format("delta") \
            .saveAsTable(dt_table_name)
        print(f"Inserted {inserted_count} new rows into '{dt_table_name}'.")
else:
    # First load: create the table with all rows
    print("Overwriting dates table")
    spark_df.write \
        .mode("overwrite") \
        .format("delta") \
        .saveAsTable(dt_table_name)
    print(f"table {dt_table_name} created")

print("dates load completed (insert-only for new keys)...")    

**Extracting entity: Capacities**

In [ ]:
item_name = "rpt_Capacities"
dt_table_name = "capacities"
report_id = items.loc[
    items["Display Name"] == item_name,
    "Id"
].iloc[0]

# build report url
url = (
    f"https://api.powerbi.com/v1.0/myorg"
    f"/reports/{report_id}"
    f"/ExportTo"
)
print(url)

# create CSV
payload = {
    "format": "CSV"
}

headers = {
    "Authorization": f"Bearer {token}"
}

response = requests.post(
    url,
    headers=headers,
    json=payload
)
result = response.json()
print(result)

# Get the export id
export_id = result["id"]
print(export_id)

# wait for export to be completed.
status_url = f"https://api.powerbi.com/v1.0/myorg/reports/{report_id}/exports/{export_id}"
while True:

    r = requests.get(
        status_url,
        headers=headers
    )

    result = r.json()
    print(result["status"])

    if result["status"] == "Succeeded":
        break

    time.sleep(10)


# download the CSV

# Resource location from your export result
download_url = f"https://api.powerbi.com/v1.0/myorg/reports/{report_id}/exports/{export_id}/file"
print(download_url)

r = requests.get(
    download_url,
    headers=headers
)
print("download status:")
print(r.status_code)

# Build dynamic filename: capacities_yyyyMMdd.csv
suffix = datetime.now().strftime("%Y%m%d")
file_name = f"Capacities_{suffix}.csv"

# Save to Lakehouse Files area
output_file = f"/lakehouse/default/Files/Capacities/{file_name}"
with open(output_file, "wb") as f:
    f.write(r.content)
print("File saved:", output_file)

# create a dataframe

df = pd.read_csv(
    output_file
)
display(df.head())

# create/append to delta table with only new Capacities_capacity_Id rows
spark_df = spark.createDataFrame(df)

key_col = "Capacities_capacity_Id"

from pyspark.sql.functions import col

if spark.catalog.tableExists(dt_table_name):
    # Load existing table and find only new keys
    print("Capacities table exist")
    existing_keys_df = spark.table(dt_table_name).select(key_col).distinct()
    new_rows_df = spark_df.join(existing_keys_df, on=key_col, how="left_anti")

    inserted_count = new_rows_df.count()
    if inserted_count == 0:
        print(f"No new rows to insert into '{dt_table_name}'.")
    else:
        new_rows_df.write \
            .mode("append") \
            .format("delta") \
            .saveAsTable(dt_table_name)
        print(f"Inserted {inserted_count} new rows into '{dt_table_name}'.")
else:    
    # First load: create the table with all rows
    print("Overwriting capacities table")
    spark_df.write \
        .mode("overwrite") \
        .format("delta") \
        .saveAsTable(dt_table_name)
    print(f"table {dt_table_name} created")

print("capacities load completed (insert-only for new keys)...")    